In [3]:
import torch
import glob
import os
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from pathlib import Path

from utils import functions as uf
from utils.model import DynamicMLP
from sklearn.model_selection import train_test_split


In [5]:
folder_path = './data/'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

id_col = "user_id"
random_seed = 42

# 1. Carichiamo tutti i file di training (*c000.csv)
csv_files = glob.glob(os.path.join(folder_path, '*c000.csv'))
df_all = pd.concat((pd.read_csv(file, sep=";") for file in csv_files), ignore_index=True)

# 2. Carichiamo il Forget Set (D_f) salvato in data/
forget_path = os.path.join(folder_path, 'forget_data.csv')
forget_df = pd.read_csv(forget_path, sep=";" if ";" in open(forget_path).readline() else ",")

# 3. Estraiamo gli utenti da dimenticare
forget_user_ids = forget_df[id_col].unique()

# 4. Dai rimanenti utenti nel dataset principale, estraiamo un Test Set (es. 10%)
non_forget_users = df_all[~df_all[id_col].isin(forget_user_ids)][id_col].unique()

retain_users, test_users = train_test_split(
    non_forget_users, 
    test_size=0.10, 
    random_state=random_seed
)

# 5. Mappiamo i DataFrame definitivi
retain_df = df_all[df_all[id_col].isin(retain_users)].reset_index(drop=True)  # D_r
test_df   = df_all[df_all[id_col].isin(test_users)].reset_index(drop=True)    # Test Set
train_df  = df_all  # Set completo originale (D_r + D_f + Test)

# Stampa di verifica
print(f"Dataset configurati con successo:")
print(f" - Train Set completo (D): {train_df.shape[0]} righe | {train_df[id_col].nunique()} utenti")
print(f" - Retain Set (D_r):       {retain_df.shape[0]} righe | {retain_df[id_col].nunique()} utenti")
print(f" - Forget Set (D_f):       {forget_df.shape[0]} righe | {forget_df[id_col].nunique()} utenti")
print(f" - Test Set (creato):      {test_df.shape[0]} righe | {test_df[id_col].nunique()} utenti")

Dataset configurati con successo:
 - Train Set completo (D): 129783 righe | 129783 utenti
 - Retain Set (D_r):       108628 righe | 108628 utenti
 - Forget Set (D_f):       9085 righe | 9085 utenti
 - Test Set (creato):      12070 righe | 12070 utenti


In [6]:
X_train, y_train, feature_cols, target_cols = uf.prepare_data(train_df, id_col=id_col, target_prefix='target__') # careful! here the train is not the real train set

imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train).astype(np.float32)



pos_counts = np.sum(y_train, axis=0)
neg_counts = len(y_train) - pos_counts
pos_weights = torch.tensor(neg_counts / (pos_counts + 1e-6), device=device)
pos_weights = pos_weights.clamp(min=0.1, max=100.0)
print(f"pos_weights: {pos_weights}")


artifact_path = Path('data') / 'model_artifact'

payload = uf.load_pickle(artifact_path)

state_dict = payload['state_dict']
architecture = payload['architecture']
best_params = payload['best_hyperparameters']
model_class_source = payload['model_class_source']

print("\n--- Saved Metadata ---")
print("Architecture parameters:", architecture)
print("Best Hyperparameters:", best_params)

try:
    model = DynamicMLP(
        input_dim=architecture['input_dim'],
        hidden_layers=architecture['hidden_layers'],
        num_outputs=architecture['num_outputs']
    )
except NameError:
    print("DynamicMLP class was not found. Check if the class source compiled correctly.")
    raise

model.load_state_dict(state_dict)

model.eval()

print("\nModel successfully reconstructed and weights loaded.")

DataFrame columns: ['ang_m_datediff_att_linea', 'ang_m_flag_mnp', 'ang_m_datediff_cessazione', 'ang_m_datediff_scad_sim', 'ang_m_cns_cre_res_per', 'ang_m_anz_linea', 'ang_m_datediff_mnp_in', 'ang_m_datediff_cam_prof', 'ang_m_datediff_mnp_out', 'ohe_ang_m_stato_linea_a', 'ohe_ang_m_stato_linea_c', 'ohe_ang_m_stato_linea_n', 'ohe_ang_m_stato_linea_s', 'ohe_ang_m_tipo_linea_fwa', 'ohe_ang_m_tipo_linea_pp', 'ohe_ang_m_tipo_linea_xt', 'ohe_ang_m_area_territoriale_altro_nd', 'ohe_ang_m_area_territoriale_c', 'ohe_ang_m_area_territoriale_ne', 'ohe_ang_m_area_territoriale_no', 'ohe_ang_m_area_territoriale_s', 'ohe_ang_m_pdv_canale_des_top_4_altro', 'ohe_ang_m_pdv_canale_des_top_4_grande_distribuzione', 'ohe_ang_m_pdv_canale_des_top_4_null', 'ohe_ang_m_pdv_canale_des_top_4_one_team', 'ohe_ang_m_pdv_canale_des_top_4_retail', 'ohe_ang_m_pdv_canale_des_top_4_tele_sales', 'ohe_ang_m_mnp_olo_prov_infrastrutturati', 'ohe_ang_m_mnp_olo_prov_mvno', 'ohe_ang_m_mnp_olo_prov_nd', 'ohe_ang_m_mnp_olo_dest_in

RuntimeError: Could not infer dtype of numpy.float32